# 07 — Construct the combined eleven-batch evaluation pool

Ensure that the processed batches share the same 304-feature representation,
concatenate them and deduplicate object identifiers. Store the virtual combined
batch as batch 99. The pool has 352,691 objects, including 1,489 anomalies.

Reuse compatible feature caches. Missing or invalid caches can trigger expensive
feature extraction; the combined catalogue is not the full raw PLAsTiCC test set.


## 0. Configuration


In [ ]:
%load_ext autoreload
%autoreload 2
import sys, os, time
from pathlib import Path
raiz = Path.cwd()
while not (raiz / 'config.py').exists() and raiz != raiz.parent:
    raiz = raiz.parent
assert (raiz / 'config.py').exists(), 'Open this notebook inside the project directory'
os.chdir(raiz); sys.path.insert(0, str(raiz))
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import config
from src import pipeline
from src.models import classification
from src.features.cache import load_cache, write_cache

BATCHES    = list(range(1, 12))      
BATCH_COMB = 99                      
print('batches:', BATCHES, '· combinado:', BATCH_COMB)

RECOMPUTE_POOL = False


## 1. Ensure features exist for all eleven batches


In [ ]:
if RECOMPUTE_POOL:
    for b in BATCHES:
        batch_paths = pipeline.select_batch(b)
        file_paths = (batch_paths.features_test_csv, batch_paths.features_gp_test_csv, batch_paths.features_gp2d_test_csv)
        if all(Path(r).exists() for r in file_paths):
            print(f'batch {b:2d}: features ya cached ✓')
            continue
        if not RECOMPUTE_POOL:
            raise FileNotFoundError(f'Missing batch-{b} caches; import them or explicitly enable RECOMPUTE_POOL.')
        print(f'batch {b:2d}: computing features from light curves (GP extraction can be slow)…')
        t = time.time()
        curvas, metadata_table = pipeline.clean_batch(batch_paths=batch_paths, verbose=True)
        _ = pipeline.features_batch(curvas, metadata_table, batch_paths=batch_paths, usar_cache=False, guardar_cache=True, verbose=True)
        print(f'   batch {b:2d} hecho in {(time.time()-t)/60:.1f} min')
    print('features of the 11 batches listas.')
else:
    print('Using the validated combined-pool caches; individual batch extraction inputs are not needed.')


## 2. Construct the virtual combined batch


In [ ]:
if RECOMPUTE_POOL:
    glob = pd.read_csv(config.TEST_METADATA_CSV, usecols=['object_id', 'true_target'])
    stats_all, gp_all, gp2d_all = [], [], []
    for b in BATCHES:
        batch_paths = pipeline.select_batch(b)
        stats_all.append(load_cache(batch_paths.features_test_csv, 'estadisticas'))
        gp_all.append(load_cache(batch_paths.features_gp_test_csv, 'gp1d'))
        gp2d_all.append(load_cache(batch_paths.features_gp2d_test_csv, 'gp2d'))
    
    Sdf  = pd.concat(stats_all, ignore_index=True).drop_duplicates('object_id')
    Gdf  = pd.concat(gp_all,   ignore_index=True).drop_duplicates('object_id')
    G2df = pd.concat(gp2d_all, ignore_index=True).drop_duplicates('object_id')
    oids = Sdf['object_id'].to_numpy()
    meta_comb = glob[glob['object_id'].isin(oids)].drop_duplicates('object_id').copy()
    
    rutas_pool = pipeline.select_batch(BATCH_COMB)
    write_cache(Sdf, rutas_pool.features_test_csv, 'estadisticas', producer_name='notebooks/07_combined_pool.ipynb')
    write_cache(Gdf, rutas_pool.features_gp_test_csv, 'gp1d', producer_name='notebooks/07_combined_pool.ipynb')
    write_cache(G2df, rutas_pool.features_gp2d_test_csv, 'gp2d', producer_name='notebooks/07_combined_pool.ipynb')
    meta_comb.to_csv(rutas_pool.test_metadata_clean_csv, index=False)
    print(f'BATCH COMBINADO {BATCH_COMB}: {len(Sdf):,} objects · '
          f'{int((meta_comb.true_target>=990).sum())} anomalies 99x · saved with sufijo _b99')
else:
    rutas_pool = pipeline.select_batch(BATCH_COMB)
    meta_comb = pd.read_csv(rutas_pool.test_metadata_clean_csv)
    if meta_comb.object_id.duplicated().any():
        raise ValueError('Duplicate combined-pool metadata identifiers')
    for cache_path, family in [(rutas_pool.features_test_csv,'estadisticas'),
                               (rutas_pool.features_gp_test_csv,'gp1d'),
                               (rutas_pool.features_gp2d_test_csv,'gp2d')]:
        cached = load_cache(cache_path, family)
        if cached.object_id.duplicated().any() or set(cached.object_id) != set(meta_comb.object_id):
            raise ValueError('Combined-pool feature and metadata identifiers disagree')
        print('Validated combined-pool cache:', cache_path.name, cached.shape)
        del cached
    print(f'Combined pool: {len(meta_comb):,} objects; {int(meta_comb.true_target.ge(990).sum()):,} anomalies.')


## 3. Cold-start classification


In [ ]:
if RECOMPUTE_POOL:
    
    tr = classification.features_e2gp('train')
    X_tr, COLS = classification.e2gp_matrix(tr); y_tr = tr['target'].to_numpy()
    batch_paths = pipeline.select_batch(BATCH_COMB)
    te = classification.features_e2gp('test', batch=BATCH_COMB)
    X_te, _ = classification.e2gp_matrix(te, COLS)
    
    pred, _ = pipeline._batch_predictions(
        X_tr, y_tr, X_te, te['object_id'].to_numpy(),
        lambda y: classification.e2gp_al(y=y, random_state=config.RANDOM_STATE))
    pred.to_csv(batch_paths.pred_test_csv, index=False)
    print('guardado:', batch_paths.pred_test_csv.name, '·', f'{len(pred):,}', 'Cold-start classified objects')
    pred.head()
else:
    pred = pd.read_csv(rutas_pool.pred_test_csv)
    if pred.object_id.duplicated().any() or set(pred.object_id) != set(meta_comb.object_id):
        raise ValueError('Combined-pool predictions and metadata identifiers disagree')
    if not pred.confianza.between(0,1).all():
        raise ValueError('Invalid saved combined-pool confidence')
    print('Reused canonical pool predictions; no classifier refitted.')
    display(pred.head())


## Outputs

This notebook produces the combined feature and metadata tables and cold-start
predictions. The ten-seed policy comparison belongs to notebook 07b.
